# AlexNet Feature Extraction

## 1. Imports

In [2]:
import os

import numpy as np
import pandas as pd
import pickle
from PIL import Image
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T


## 2. Load the pretrained model

`eval()` mode disables dropout and freezes batch-norm statistics, so the same image always produces the same activations.

In [4]:
model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(model)

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

## 3. Register forward hooks on every layer

In [5]:
activations = {}

def make_hook(layer_name):
    def hook(module, input, output):
        activations[layer_name] = output.detach().cpu().numpy().reshape(output.shape[0], -1)
    return hook

In [7]:
hooks = []
layer_names = []

# convolutional
for idx, layer in enumerate(model.features):
    if isinstance(layer, nn.ReLU):
        name = f"features_{idx}_relu"
        hooks.append(layer.register_forward_hook(make_hook(name)))
        layer_names.append(name)

# classifier
for idx, layer in enumerate(model.classifier):
    if isinstance(layer, nn.Linear) and idx == 6:
        name = f"classifier_{idx}_linear_output"
        hooks.append(layer.register_forward_hook(make_hook(name)))
        layer_names.append(name)
    elif isinstance(layer, nn.ReLU):
        name = f"classifier_{idx}_relu"
        hooks.append(layer.register_forward_hook(make_hook(name)))
        layer_names.append(name)

print(f"Hooked {len(layer_names)} layers:")
for n in layer_names:
    print(" -", n)


Hooked 8 layers:
 - features_1_relu
 - features_4_relu
 - features_7_relu
 - features_9_relu
 - features_11_relu
 - classifier_2_relu
 - classifier_5_relu
 - classifier_6_linear_output


## 4. Image preprocessing

Images are resized to 224x224 and converted from single-channel grayscale to 3-channel RGB (AlexNet expects 3 input channels) and normalized with standard ImageNet statistics.

In [8]:
preprocess = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def preprocess_image(img_array):
    arr = np.asarray(img_array)
    if arr.dtype != np.uint8:
        arr = arr.astype(np.float32)
        arr = arr - arr.min()
        if arr.max() > 0:
            arr = arr / arr.max()
        arr = (arr * 255).astype(np.uint8)
    return preprocess(arr)


## 5. Load images

Uses `image_labels.csv` to fix the image order.

In [9]:
data_directory = "./ecephys_cache_dir"
IMAGE_DIR = os.path.join(data_directory, "natural_scene_templates")

labels_df = pd.read_csv("image_labels.csv")
image_names = labels_df["image_name"].tolist()

images = {
    name: np.array(Image.open(os.path.join(IMAGE_DIR, f"{name}.png")))
    for name in image_names
}

assert len(images) == 118, f"Expected 118 images, found {len(images)}"

## 6. Run all images through AlexNet and collect activations

In [10]:
all_layer_activations = {name: [] for name in layer_names}

with torch.no_grad():
    for name in image_names:
        img_tensor = preprocess_image(images[name]).unsqueeze(0).to(device)
        model(img_tensor)
        for layer_name in layer_names:
            all_layer_activations[layer_name].append(activations[layer_name][0])

final_activations = {
    name: np.stack(all_layer_activations[name], axis=0)
    for name in layer_names
}

for name, arr in final_activations.items():
    print(f"{name}: shape {arr.shape}")

features_1_relu: shape (118, 193600)
features_4_relu: shape (118, 139968)
features_7_relu: shape (118, 64896)
features_9_relu: shape (118, 43264)
features_11_relu: shape (118, 43264)
classifier_2_relu: shape (118, 4096)
classifier_5_relu: shape (118, 4096)
classifier_6_linear_output: shape (118, 1000)


## 7. Save activations 

In [11]:
with open("alexnet_activations.pkl", "wb") as f:
    pickle.dump({
        "image_names": image_names,
        "activations": final_activations,
    }, f)

print("Saved alexnet_activations.pkl")

for h in hooks:
    h.remove()

Saved alexnet_activations.pkl
